In [ ]:
import torch
from torchinfo import summary
from torch.utils.data import DataLoader
import pandas as pd
import os
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
from tqdm.notebook import tqdm

from dotenv import load_dotenv
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, f1_score, classification_report

from src.transformer.transformer_architecture import KeypointTransformer, LitKeypointTransformer
from src.image_processing.mediapipe import MediaPipeProcessor
from src.image_processing.dataset_handler import DatasetHandler
from src.datasets.datasets import IPNData, MontalbanoData

In [ ]:
load_dotenv()
IPN_GESTURE_VIDEOS = os.getenv("IPN_GESTURE_VIDEOS")
MONTALBANO_GESTURE_VIDEOS = os.getenv("MONTALBANO_GESTURE_VIDEOS")

In [ ]:
confg = {
    "IPN": {
        "dataset": IPNData(source_path=IPN_GESTURE_VIDEOS),
        "processor": MediaPipeProcessor(),
        "save_directory": "data/ipn/dataset.pkl"
    },
    "MONTALBANO": {
        "dataset": MontalbanoData(source_path=MONTALBANO_GESTURE_VIDEOS, target_path=os.path.join(MONTALBANO_GESTURE_VIDEOS, "target_folder")),
        "processor": MediaPipeProcessor(),
        "save_directory": "data/montalbano/dataset.pkl"
    },
}

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774381758.360264   12102 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774381758.371281   12102 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


TypeError: MontalbanoData.__init__() missing 1 required positional argument: 'target_path'

In [ ]:
ds_handler = DatasetHandler(confg)

In [ ]:
ds_handler.process(["IPN", "MONTALBANO"])

In [ ]:
ds_handler.print_statistics(["IPN", "MONTALBANO"])

In [ ]:
num_videos = ds_handler.get_num_videos(["IPN", "MONTALBANO"])
num_labels = ds_handler.get_num_labels(["IPN", "MONTALBANO"])

test_size_ipn = int(0.1*num_videos("IPN"))
train_val_size_ipn = num_videos("IPN") - test_size_ipn
train_size_ipn = int(0.9*train_val_size_ipn)
val_size_ipn = train_val_size_ipn - train_size_ipn

test_size_montalbano = int(0.1*num_videos("MOBTALBANO"))
train_val_size_montalbano = num_videos("MONTALBANO") - test_size_montalbano
train_size_montalbano = int(0.9*train_val_size_montalbano)
val_size_montalbano = train_val_size_montalbano - train_size_montalbano

In [ ]:
train_ds_ipn, val_ds_ipn, test_ds_ipn = ds_handler.get_split_ds(dataset_name="IPN", train_size=train_size_ipn, val_size=val_size_ipn, test_size=test_size_ipn, shuffle=True)
train_ds_montalbano, val_ds_montalbano, test_ds_montalbano = ds_handler.get_split_ds(dataset_name="MONTALBANO", train_size=train_size_montalbano, val_size=val_size_montalbano, test_size=test_size_montalbano, shuffle=True)

In [31]:
def collate_fn(batch):
    sequences, labels = zip(*batch)

    sequences = [torch.tensor(s, dtype=torch.float32) for s in sequences]

    # Padding
    padded_sequences = pad_sequence(sequences, batch_first=True)

    # Masking
    mask = torch.zeros(size=padded_sequences.shape[:2], dtype=torch.bool)
    for(i, s) in enumerate(sequences):
        mask[i, :len(s)] = True

    # Labeling
    labels = torch.tensor([int(l) for l in labels])

    return padded_sequences, mask, labels

In [ ]:
train_loader = DataLoader(train_ds_ipn, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds_ipn, batch_size=32, collate_fn=collate_fn)

input_size = 126
d_model = 132

num_classes = ds_handler.get

model = LitKeypointTransformer(input_size=input_size, d_model=d_model, num_classes=num_classes)

early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min")

trainer = L.Trainer(
    max_epochs=60,
    accelerator="auto",
    devices=1,
    callbacks=[early_stopping]
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [33]:
x_batch, mask, y_batch = next(iter(train_loader))
logits = model(x_batch)  
print("x:", x_batch.shape, "y:", y_batch.shape, "logits:", logits.shape)

x: torch.Size([32, 538, 126]) y: torch.Size([32]) logits: torch.Size([32, 14])


In [34]:
trainer.fit(model, train_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type                | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | model   | KeypointTransformer | 1.3 M  | train | 0    
1 | loss_fn | CrossEntropyLoss    | 0      | train | 0    
----------------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params
5.033     Total estimated model params size (MB)
91        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [35]:
from pathlib import Path
save_direct = Path("checkpoints")
trainer.save_checkpoint(save_direct / "keypoint_transformer_extensive.ckpt") 

`weights_only` was not set, defaulting to `False`.


In [36]:
model.eval()

LitKeypointTransformer(
  (model): KeypointTransformer(
    (input_proj): Sequential(
      (0): Linear(in_features=126, out_features=132, bias=True)
    )
    (pos_encode): SinusoidalPositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncodingLayer(
        (attention): MultiHeadAttention(
          (w_q): Linear(in_features=132, out_features=132, bias=True)
          (w_k): Linear(in_features=132, out_features=132, bias=True)
          (w_v): Linear(in_features=132, out_features=132, bias=True)
          (output): Linear(in_features=132, out_features=132, bias=True)
        )
        (norm1): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
        (ffn): Sequential(
          (0): Linear(in_features=132, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_f

In [ ]:
#Testing model

In [37]:
all_labels = []
all_preds = []

for i in tqdm(range(len(test_ds))):    
    keypoint_sequence, label = test_ds[i]

    x = torch.tensor(keypoint_sequence, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, 126)

    # prediction
    with torch.no_grad():
        logits = model(x)
        predicted_idx = torch.argmax(logits, dim=-1).item()

    all_labels.append(label)
    all_preds.append(predicted_idx)


  0%|          | 0/564 [00:00<?, ?it/s]

In [38]:
# F1 score and accuracy (again)
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="macro")
class_report = classification_report(all_labels, all_preds)

print(accuracy)
print(f1)
print(class_report)

0.8191489361702128
0.732842189896462
              precision    recall  f1-score   support

           0       0.87      0.92      0.89       148
           1       0.88      0.95      0.91       101
           2       0.94      0.85      0.89        98
           3       0.55      0.32      0.40        19
           4       0.83      0.50      0.62        20
           5       0.64      0.80      0.71        20
           6       0.78      0.70      0.74        20
           7       0.87      0.68      0.76        19
           8       0.84      0.80      0.82        20
           9       0.60      0.75      0.67        20
          10       0.46      0.65      0.54        20
          11       0.59      0.65      0.62        20
          12       1.00      0.85      0.92        20
          13       0.78      0.74      0.76        19

    accuracy                           0.82       564
   macro avg       0.76      0.73      0.73       564
weighted avg       0.83      0.82      0.82